# Predicting Stellar Class
## Score: .96656


In [103]:
OVERNIGHT_MODE = True

if OVERNIGHT_MODE:
    N_SPLITS = 10
    SEEDS = [42, 1337]
    LEARNING_RATE = 0.03
    NUM_BOOST_ROUND = 10000
    EARLY_STOPPING_ROUNDS = 150
    XGB_N_SPLITS = 10
    XGB_SEEDS = [42, 1337]
    XGB_LR = 0.03
    XGB_NUM_ROUNDS = 10000
    XGB_EARLY_STOPPING = 150
    CB_N_SPLITS = 5
    CB_SEEDS = [42]
    CB_ITERATIONS = 8000
    CB_LR = 0.04
    LGB_CONFIGS = [
        ("lgb", {"num_leaves": 63}),
        ("lgb_mid", {"num_leaves": 95}),
        ("lgb_deep", {"num_leaves": 127}),
    ]
    EST_LGB_FITS = N_SPLITS * len(SEEDS) * len(LGB_CONFIGS)
    EST_XGB_FITS = XGB_N_SPLITS * len(XGB_SEEDS)
    EST_CB_FITS = CB_N_SPLITS * len(CB_SEEDS)
    print(f"overnight: ~{EST_LGB_FITS} LGB + {EST_XGB_FITS} XGB + {EST_CB_FITS} CB fits (was ~140)")
else:
    N_SPLITS = 5
    SEEDS = [42, 1337]
    LEARNING_RATE = 0.05
    NUM_BOOST_ROUND = 5000
    EARLY_STOPPING_ROUNDS = 100
    XGB_N_SPLITS = 5
    XGB_SEEDS = [42, 1337]
    XGB_LR = 0.05
    XGB_NUM_ROUNDS = 5000
    XGB_EARLY_STOPPING = 100
    CB_N_SPLITS = 0
    CB_SEEDS = []
    CB_ITERATIONS = 0
    CB_LR = 0.0
    LGB_CONFIGS = [
        ("lgb", {"num_leaves": 63}),
        ("lgb_mid", {"num_leaves": 95}),
        ("lgb_deep", {"num_leaves": 127}),
    ]
    EST_LGB_FITS = N_SPLITS * len(SEEDS) * len(LGB_CONFIGS)
    EST_XGB_FITS = XGB_N_SPLITS * len(XGB_SEEDS)
    print(f"fast: {EST_LGB_FITS} LGB + {EST_XGB_FITS} XGB fits (2-seed bagging)")

overnight: ~60 LGB + 20 XGB + 5 CB fits (was ~140)


In [104]:
import numpy as np
import pandas as pd

DATA_DIR = "playground-series-s6e6"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"train: {train.shape}    test: {test.shape}")
print()
print("dtypes:")
print(train.dtypes)
print()

print("class distribution (train):")
print(train["class"].value_counts(normalize=True).round(4).to_string())
print()

nulls = train.isna().sum()
print("nulls per column (train):")
print(nulls[nulls > 0].to_string() if nulls.any() else "  none")
print()

print("categorical cardinalities:")
for c in ["spectral_type", "galaxy_population"]:
    vals = sorted(train[c].unique().tolist())
    print(f"  {c}: {len(vals)} unique  ->  {vals}")
print()

print("numeric summary:")
train.drop(columns=["id"]).describe().T

train: (577347, 12)    test: (247435, 11)

dtypes:
id                     int64
alpha                float64
delta                float64
u                    float64
g                    float64
r                    float64
i                    float64
z                    float64
redshift             float64
spectral_type         object
galaxy_population     object
class                 object
dtype: object

class distribution (train):
class
GALAXY    0.6538
QSO       0.2029
STAR      0.1433

nulls per column (train):
  none

categorical cardinalities:
  spectral_type: 4 unique  ->  ['A/F', 'G/K', 'M', 'O/B']
  galaxy_population: 2 unique  ->  ['Blue_Cloud', 'Red_Sequence']

numeric summary:


,count,mean,std,min,25%,50%,75%,max
alpha,577347.0,181.616673,96.242941,0.011684,132.161499,188.681465,231.829693,359.999810
delta,577347.0,21.834654,18.933570,-17.966988,2.474097,21.484412,36.988310,79.158322
u,577347.0,22.441926,2.018135,-0.139225,20.977090,22.570222,23.869103,28.253263
g,577347.0,21.007273,1.795426,13.535483,19.865005,21.467820,22.292715,27.620208
r,577347.0,19.962811,1.648964,12.579407,18.820671,20.431153,21.164096,25.254499
i,577347.0,19.378911,1.580059,11.962781,18.306820,19.631642,20.608191,27.910853
z,577347.0,19.041136,1.584365,11.682803,17.973192,19.188598,20.162111,26.826867
redshift,577347.0,0.723135,0.810070,-0.009970,0.181052,0.497525,0.881390,7.010780


In [105]:
DROP_FEATURES = ["spectral_type", "galaxy_population"]
BASE_FEATURES = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift", "spectral_type", "galaxy_population"]
CAT_FEATURES = ["spectral_type", "galaxy_population"]
BASE_FEATURES = [c for c in BASE_FEATURES if c not in DROP_FEATURES]
CAT_FEATURES = [c for c in CAT_FEATURES if c not in DROP_FEATURES]
print(f"dropped features: {DROP_FEATURES}")
TARGET = "class"
LABELS = ["GALAXY", "QSO", "STAR"]
label_to_idx = {l: i for i, l in enumerate(LABELS)}
idx_to_label = {i: l for l, i in label_to_idx.items()}

MAGNITUDE_COLS = ["u", "g", "r", "i", "z"]
SENTINELS = {-9999.0, -999.0, -99.0, 99.0, 999.0, 9999.0}

def clean_sentinels(df):
    df = df.copy()
    cleaned = 0
    for c in MAGNITUDE_COLS + ["redshift"]:
        mask = df[c].isin(SENTINELS) | (df[c] > 50) | (df[c] < -50)
        if mask.any():
            cleaned += int(mask.sum())
            df.loc[mask, c] = np.nan
    return df, cleaned

def add_color_features(df, extended=False):
    df = df.copy()
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    df["u_r"] = df["u"] - df["r"]
    df["g_i"] = df["g"] - df["i"]
    df["u_z"] = df["u"] - df["z"]
    if extended:
        df["u_i"] = df["u"] - df["i"]
        df["g_z"] = df["g"] - df["z"]
        df["r_z"] = df["r"] - df["z"]
        df["mag_mean"] = df[MAGNITUDE_COLS].mean(axis=1)
        df["mag_std"] = df[MAGNITUDE_COLS].std(axis=1)
        df["log1p_redshift"] = np.sign(df["redshift"]) * np.log1p(np.abs(df["redshift"]))
        df["u_g_sq"] = df["u_g"] ** 2
        df["g_r_sq"] = df["g_r"] ** 2
    return df

train_fe, train_cleaned = clean_sentinels(train)
test_fe, test_cleaned = clean_sentinels(test)
print(f"sentinel cleaning -> train: {train_cleaned} cells nan'd    test: {test_cleaned} cells nan'd")

train_fe = add_color_features(train_fe, extended=OVERNIGHT_MODE)
test_fe = add_color_features(test_fe, extended=OVERNIGHT_MODE)

COLOR_FEATURES = ["u_g", "g_r", "r_i", "i_z", "u_r", "g_i", "u_z"]
EXTENDED_FEATURES = ["u_i", "g_z", "r_z", "mag_mean", "mag_std", "log1p_redshift", "u_g_sq", "g_r_sq"]
ALL_FEATURES = BASE_FEATURES + COLOR_FEATURES + (EXTENDED_FEATURES if OVERNIGHT_MODE else [])

for c in CAT_FEATURES:
    train_fe[c] = train_fe[c].astype("category")
    test_fe[c] = test_fe[c].astype("category")

X = train_fe[ALL_FEATURES]
y = train_fe[TARGET].map(label_to_idx).astype(int)
X_test = test_fe[ALL_FEATURES]

print(f"X: {X.shape}    y: {y.shape}    X_test: {X_test.shape}")
print(f"features ({len(ALL_FEATURES)}): {ALL_FEATURES}")

dropped features: ['spectral_type', 'galaxy_population']
sentinel cleaning -> train: 0 cells nan'd    test: 0 cells nan'd
X: (577347, 23)    y: (577347,)    X_test: (247435, 23)
features (23): ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'u_z', 'u_i', 'g_z', 'r_z', 'mag_mean', 'mag_std', 'log1p_redshift', 'u_g_sq', 'g_r_sq']


In [106]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

freq = np.bincount(y.to_numpy(), minlength=len(LABELS)) / len(y)
inv_freq = 1.0 / freq

WEIGHT_ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
PER_CLASS_ALPHAS = [0.75, 1.0, 1.25, 1.5, 1.75, 2.0]

tune_params = {
    "objective": "multiclass",
    "num_class": len(LABELS),
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 200,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "verbose": -1,
    "seed": 42,
}

alpha_skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

def eval_alpha_oof(alphas):
    vec = inv_freq ** np.array(alphas, dtype=np.float64)
    vec = vec / vec.mean()
    oof_alpha = np.zeros((len(X), 3))
    for tr_idx, va_idx in alpha_skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        w_tr = vec[y_tr.to_numpy()]
        dtr = lgb.Dataset(X_tr, y_tr, weight=w_tr, categorical_feature=CAT_FEATURES)
        dva = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtr)
        m = lgb.train(
            tune_params, dtr, num_boost_round=2000, valid_sets=[dva],
            callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)],
        )
        oof_alpha[va_idx] = m.predict(X_va, num_iteration=m.best_iteration)
    score = balanced_accuracy_score(y, oof_alpha.argmax(axis=1))
    return score, vec

best_alphas = np.array([1.0, 1.0, 1.0])
best_score, class_weight_vec = -1.0, None

for alpha in WEIGHT_ALPHAS:
    s, vec = eval_alpha_oof([alpha, alpha, alpha])
    print(f"alpha={alpha:.2f}: OOF balanced_acc = {s:.5f}    weights = {np.round(vec, 3)}")
    if s > best_score:
        best_score, best_alphas, class_weight_vec = s, np.array([alpha, alpha, alpha]), vec

print(f"\nunified best alpha: {best_alphas[0]:.2f}    OOF balanced_acc: {best_score:.5f}")

for _ in range(2):
    for cls_idx, cls_name in enumerate(LABELS):
        for a in PER_CLASS_ALPHAS:
            trial = best_alphas.copy()
            trial[cls_idx] = a
            s, vec = eval_alpha_oof(trial)
            if s > best_score:
                best_score, best_alphas, class_weight_vec = s, trial, vec
        print(
            f"  {cls_name}: alpha={best_alphas[cls_idx]:.2f}    "
            f"alphas={np.round(best_alphas, 2)}    weights={np.round(class_weight_vec, 3)}    OOF={best_score:.5f}"
        )

print(f"\nbest alphas: {dict(zip(LABELS, np.round(best_alphas, 2)))}")
print(f"class weights: {np.round(class_weight_vec, 4)}    OOF balanced_acc: {best_score:.5f}")

alpha=0.00: OOF balanced_acc = 0.95686    weights = [1. 1. 1.]
alpha=0.25: OOF balanced_acc = 0.95986    weights = [0.789 1.057 1.153]
alpha=0.50: OOF balanced_acc = 0.96204    weights = [0.608 1.092 1.3  ]
alpha=0.75: OOF balanced_acc = 0.96332    weights = [0.46  1.105 1.435]
alpha=1.00: OOF balanced_acc = 0.96391    weights = [0.341 1.1   1.558]
alpha=1.25: OOF balanced_acc = 0.96436    weights = [0.25  1.081 1.669]
alpha=1.50: OOF balanced_acc = 0.96484    weights = [0.181 1.05  1.769]

unified best alpha: 1.50    OOF balanced_acc: 0.96484
  GALAXY: alpha=0.75    alphas=[0.75 1.5  1.5 ]    weights=[0.134 1.067 1.799]    OOF=0.96516
  QSO: alpha=1.50    alphas=[0.75 1.5  1.5 ]    weights=[0.134 1.067 1.799]    OOF=0.96516
  STAR: alpha=1.75    alphas=[0.75 1.5  1.75]    weights=[0.098 0.776 2.126]    OOF=0.96535
  GALAXY: alpha=0.75    alphas=[0.75 1.5  1.75]    weights=[0.098 0.776 2.126]    OOF=0.96535
  QSO: alpha=1.50    alphas=[0.75 1.5  1.75]    weights=[0.098 0.776 2.126]    

In [107]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

TOTAL_FITS = N_SPLITS * len(SEEDS)

base_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": LEARNING_RATE,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "min_data_in_leaf": 200,
    "verbose": -1,
}

oof_by_lgb = {n: np.zeros((len(X), 3)) for n, _ in LGB_CONFIGS}
test_by_lgb = {n: np.zeros((len(X_test), 3)) for n, _ in LGB_CONFIGS}
fi_lgb = np.zeros(len(ALL_FEATURES))
fit_idx = 0
total_fits = TOTAL_FITS * len(LGB_CONFIGS)
print(f"LGB: {total_fits} fits ({N_SPLITS} folds x {len(SEEDS)} seeds x {len(LGB_CONFIGS)} configs)")

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    for name, cfg in LGB_CONFIGS:
        seed_oof = np.zeros((len(X), 3))
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            fit_idx += 1
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            w_tr = class_weight_vec[y_tr.to_numpy()]

            dtrain = lgb.Dataset(X_tr, y_tr, weight=w_tr, categorical_feature=CAT_FEATURES)
            dvalid = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtrain)

            params = {**base_params, **cfg, "seed": seed}
            model = lgb.train(
                params,
                dtrain,
                num_boost_round=NUM_BOOST_ROUND,
                valid_sets=[dvalid],
                callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(0)],
            )

            va_pred = model.predict(X_va, num_iteration=model.best_iteration)
            seed_oof[va_idx] = va_pred
            test_by_lgb[name] += model.predict(X_test, num_iteration=model.best_iteration)
            if name == "lgb":
                fi_lgb += model.feature_importance(importance_type="gain")

            fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
            print(f"{name} fold {fit_idx}/{total_fits}: balanced_acc = {fold_score:.5f}    best_iter = {model.best_iteration}")

        oof_by_lgb[name] += seed_oof

for name, _ in LGB_CONFIGS:
    oof_by_lgb[name] /= len(SEEDS)
    test_by_lgb[name] /= TOTAL_FITS

oof_lgb = oof_by_lgb["lgb"]
test_pred_lgb = test_by_lgb["lgb"]
oof_lgb_mid = oof_by_lgb["lgb_mid"]
test_pred_lgb_mid = test_by_lgb["lgb_mid"]
oof_lgb_deep = oof_by_lgb["lgb_deep"]
test_pred_lgb_deep = test_by_lgb["lgb_deep"]
fi_lgb /= TOTAL_FITS

for name, _ in LGB_CONFIGS:
    s = balanced_accuracy_score(y, oof_by_lgb[name].argmax(axis=1))
    print(f"{name} OOF balanced_acc: {s:.5f}")

LGB: 60 fits (10 folds x 2 seeds x 3 configs)
lgb fold 1/60: balanced_acc = 0.96547    best_iter = 4390
lgb fold 2/60: balanced_acc = 0.96545    best_iter = 4790
lgb fold 3/60: balanced_acc = 0.96580    best_iter = 4295
lgb fold 4/60: balanced_acc = 0.96494    best_iter = 4400
lgb fold 5/60: balanced_acc = 0.96462    best_iter = 4685
lgb fold 6/60: balanced_acc = 0.96491    best_iter = 4583
lgb fold 7/60: balanced_acc = 0.96528    best_iter = 4380
lgb fold 8/60: balanced_acc = 0.96467    best_iter = 4255
lgb fold 9/60: balanced_acc = 0.96412    best_iter = 4550
lgb fold 10/60: balanced_acc = 0.96612    best_iter = 4719
lgb_mid fold 11/60: balanced_acc = 0.96569    best_iter = 3018
lgb_mid fold 12/60: balanced_acc = 0.96581    best_iter = 2776
lgb_mid fold 13/60: balanced_acc = 0.96499    best_iter = 3198
lgb_mid fold 14/60: balanced_acc = 0.96505    best_iter = 2956
lgb_mid fold 15/60: balanced_acc = 0.96468    best_iter = 3085
lgb_mid fold 16/60: balanced_acc = 0.96462    best_iter = 

In [108]:
oof_cb = None
test_pred_cb = None

if OVERNIGHT_MODE:
    try:
        from catboost import CatBoostClassifier
        _cb_available = True
    except ImportError:
        print("CatBoost not installed. Run: pip install catboost")
        print("Skipping CatBoost stage.")
        _cb_available = False

    if _cb_available:
        CB_TOTAL_FITS = CB_N_SPLITS * len(CB_SEEDS)
        print(f"CatBoost: {CB_TOTAL_FITS} fits")

        cat_idx = [ALL_FEATURES.index(c) for c in CAT_FEATURES]

        X_cb = X.copy()
        X_test_cb = X_test.copy()
        for c in CAT_FEATURES:
            X_cb[c] = X_cb[c].astype(str)
            X_test_cb[c] = X_test_cb[c].astype(str)

        oof_cb = np.zeros((len(X), 3))
        test_pred_cb = np.zeros((len(X_test), 3))
        fit_idx = 0

        for seed in CB_SEEDS:
            skf = StratifiedKFold(n_splits=CB_N_SPLITS, shuffle=True, random_state=seed)
            for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb, y), start=1):
                fit_idx += 1
                X_tr, X_va = X_cb.iloc[tr_idx], X_cb.iloc[va_idx]
                y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

                model = CatBoostClassifier(
                    iterations=CB_ITERATIONS,
                    learning_rate=CB_LR,
                    depth=7,
                    l2_leaf_reg=3.0,
                    loss_function="MultiClass",
                    eval_metric="MultiClass",
                    class_weights=list(class_weight_vec),
                    early_stopping_rounds=150,
                    thread_count=-1,
                    random_seed=seed,
                    verbose=0,
                )
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_idx)

                va_pred = model.predict_proba(X_va)
                oof_cb[va_idx] += va_pred / len(CB_SEEDS)
                test_pred_cb += model.predict_proba(X_test_cb) / CB_TOTAL_FITS

                fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
                print(f"cb fold {fit_idx}/{CB_TOTAL_FITS}: balanced_acc = {fold_score:.5f}    best_iter = {model.tree_count_}")

        cb_oof_score = balanced_accuracy_score(y, oof_cb.argmax(axis=1))
        print(f"\nCatBoost OOF balanced_acc: {cb_oof_score:.5f}")

CatBoost: 5 fits
cb fold 1/5: balanced_acc = 0.95528    best_iter = 2770
cb fold 2/5: balanced_acc = 0.95391    best_iter = 2359
cb fold 3/5: balanced_acc = 0.95465    best_iter = 2578
cb fold 4/5: balanced_acc = 0.95454    best_iter = 2223
cb fold 5/5: balanced_acc = 0.95415    best_iter = 2582

CatBoost OOF balanced_acc: 0.95451


In [109]:
oof_xgb = None
test_pred_xgb = None

try:
    import xgboost as xgb
    _xgb_available = True
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    print("Skipping XGBoost stage.")
    _xgb_available = False

if _xgb_available:
    XGB_TOTAL_FITS = XGB_N_SPLITS * len(XGB_SEEDS)
    print(f"XGB: {XGB_TOTAL_FITS} fits ({XGB_N_SPLITS} folds x {len(XGB_SEEDS)} seeds)")

    xgb_params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "max_depth": 8,
        "learning_rate": XGB_LR,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "min_child_weight": 5,
        "reg_lambda": 1.0,
        "verbosity": 0,
    }

    dtest = xgb.DMatrix(X_test, enable_categorical=True)

    oof_xgb = np.zeros((len(X), 3))
    test_pred_xgb = np.zeros((len(X_test), 3))
    fit_idx = 0

    for seed in XGB_SEEDS:
        skf = StratifiedKFold(n_splits=XGB_N_SPLITS, shuffle=True, random_state=seed)
        seed_oof = np.zeros((len(X), 3))

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            fit_idx += 1
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            w_tr = class_weight_vec[y_tr.to_numpy()]

            dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr, enable_categorical=True)
            dvalid = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)

            params = {**xgb_params, "seed": seed}
            model = xgb.train(
                params,
                dtrain,
                num_boost_round=XGB_NUM_ROUNDS,
                evals=[(dvalid, "valid")],
                early_stopping_rounds=XGB_EARLY_STOPPING,
                verbose_eval=False,
            )

            best_it = model.best_iteration
            va_pred = model.predict(dvalid, iteration_range=(0, best_it + 1))
            seed_oof[va_idx] = va_pred
            test_pred_xgb += model.predict(dtest, iteration_range=(0, best_it + 1))

            fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
            print(f"xgb fold {fit_idx}/{XGB_TOTAL_FITS}: balanced_acc = {fold_score:.5f}    best_iter = {best_it}")

        oof_xgb += seed_oof

    oof_xgb /= len(XGB_SEEDS)
    test_pred_xgb /= XGB_TOTAL_FITS

    xgb_oof_score = balanced_accuracy_score(y, oof_xgb.argmax(axis=1))
    print(f"\nXGBoost OOF balanced_acc: {xgb_oof_score:.5f}")

XGB: 20 fits (10 folds x 2 seeds)
xgb fold 1/20: balanced_acc = 0.96439    best_iter = 8266
xgb fold 2/20: balanced_acc = 0.96438    best_iter = 8033
xgb fold 3/20: balanced_acc = 0.96421    best_iter = 7670
xgb fold 4/20: balanced_acc = 0.96370    best_iter = 8234
xgb fold 5/20: balanced_acc = 0.96371    best_iter = 7711
xgb fold 6/20: balanced_acc = 0.96358    best_iter = 7787
xgb fold 7/20: balanced_acc = 0.96448    best_iter = 8003
xgb fold 8/20: balanced_acc = 0.96288    best_iter = 7434
xgb fold 9/20: balanced_acc = 0.96328    best_iter = 7503
xgb fold 10/20: balanced_acc = 0.96502    best_iter = 8676
xgb fold 11/20: balanced_acc = 0.96354    best_iter = 7900
xgb fold 12/20: balanced_acc = 0.96374    best_iter = 8036
xgb fold 13/20: balanced_acc = 0.96352    best_iter = 7953
xgb fold 14/20: balanced_acc = 0.96498    best_iter = 8058
xgb fold 15/20: balanced_acc = 0.96371    best_iter = 7726
xgb fold 16/20: balanced_acc = 0.96348    best_iter = 8020
xgb fold 17/20: balanced_acc = 

In [111]:
from itertools import product

base_models = {
    "lgb": (oof_lgb, test_pred_lgb),
    "lgb_mid": (oof_lgb_mid, test_pred_lgb_mid),
    "lgb_deep": (oof_lgb_deep, test_pred_lgb_deep),
}
if oof_cb is not None:
    base_models["cb"] = (oof_cb, test_pred_cb)
if oof_xgb is not None:
    base_models["xgb"] = (oof_xgb, test_pred_xgb)

model_names = list(base_models.keys())
oof_stack = [base_models[m][0] for m in model_names]
test_stack = [base_models[m][1] for m in model_names]

def simplex_weights(n, step=0.05):
    vals = [round(float(v), 4) for v in np.arange(0, 1 + 1e-9, step)]
    combos = []

    def build(k, rem, prefix):
        if k == 1:
            w = round(rem, 4)
            if w >= -1e-9:
                combos.append(tuple(prefix + [w]))
            return
        for v in vals:
            if v <= rem + 1e-9:
                build(k - 1, round(rem - v, 4), prefix + [v])

    build(n, 1.0, [])
    return combos

def simplex_weights_near(n, center, step=0.01, radius=0.05):
    center = list(center)
    if n == 1:
        return [(1.0,)]
    grids = [
        np.arange(max(0.0, center[i] - radius), min(1.0, center[i] + radius) + 1e-9, step)
        for i in range(n - 1)
    ]
    combos = []

    def build(k, rem, prefix):
        if k == 1:
            w = round(rem, 4)
            if w >= -1e-9:
                combos.append(tuple(prefix + [w]))
            return
        for v in grids[len(prefix)]:
            if v <= rem + 1e-9:
                build(k - 1, round(rem - float(v), 4), prefix + [round(float(v), 4)])

    build(n, 1.0, [])
    return combos

def blend_stack(stack, weights_by_class):
    n_classes = stack[0].shape[1]
    out = np.zeros_like(stack[0])
    for c in range(n_classes):
        for i, arr in enumerate(stack):
            out[:, c] += weights_by_class[i, c] * arr[:, c]
    out /= out.sum(axis=1, keepdims=True)
    return out

best_weights = None
best_blend_score = -1.0
for w in simplex_weights(len(model_names), step=0.05):
    blend = sum(wi * o for wi, o in zip(w, oof_stack))
    s = balanced_accuracy_score(y, blend.argmax(axis=1))
    if s > best_blend_score:
        best_blend_score, best_weights = s, w
for w in simplex_weights_near(len(model_names), best_weights, step=0.01, radius=0.05):
    blend = sum(wi * o for wi, o in zip(w, oof_stack))
    s = balanced_accuracy_score(y, blend.argmax(axis=1))
    if s > best_blend_score:
        best_blend_score, best_weights = s, w

global_w = np.array(best_weights, dtype=np.float64)
W = np.tile(global_w[:, None], (1, len(LABELS)))
per_class_score = best_blend_score
for cls_idx in range(len(LABELS)):
    best_col = W[:, cls_idx].copy()
    best_col_score = per_class_score
    for w in simplex_weights(len(model_names), step=0.05):
        trial = W.copy()
        trial[:, cls_idx] = w
        s = balanced_accuracy_score(y, blend_stack(oof_stack, trial).argmax(axis=1))
        if s > best_col_score:
            best_col_score, best_col = s, np.array(w, dtype=np.float64)
    for w in simplex_weights_near(len(model_names), best_col, step=0.01, radius=0.05):
        trial = W.copy()
        trial[:, cls_idx] = w
        s = balanced_accuracy_score(y, blend_stack(oof_stack, trial).argmax(axis=1))
        if s > best_col_score:
            best_col_score, best_col = s, np.array(w, dtype=np.float64)
    W[:, cls_idx] = best_col
    per_class_score = best_col_score

for cls_idx in range(len(LABELS)):
    best_col = W[:, cls_idx].copy()
    best_col_score = per_class_score
    for w in simplex_weights(len(model_names), step=0.05):
        trial = W.copy()
        trial[:, cls_idx] = w
        s = balanced_accuracy_score(y, blend_stack(oof_stack, trial).argmax(axis=1))
        if s > best_col_score:
            best_col_score, best_col = s, np.array(w, dtype=np.float64)
    for w in simplex_weights_near(len(model_names), best_col, step=0.01, radius=0.05):
        trial = W.copy()
        trial[:, cls_idx] = w
        s = balanced_accuracy_score(y, blend_stack(oof_stack, trial).argmax(axis=1))
        if s > best_col_score:
            best_col_score, best_col = s, np.array(w, dtype=np.float64)
    W[:, cls_idx] = best_col
    per_class_score = best_col_score

use_per_class = per_class_score >= best_blend_score
if use_per_class:
    oof_blend = blend_stack(oof_stack, W)
    test_pred_blend = blend_stack(test_stack, W)
    best_blend_score = per_class_score
else:
    oof_blend = sum(wi * o for wi, o in zip(best_weights, oof_stack))
    test_pred_blend = sum(wi * t for wi, t in zip(best_weights, test_stack))

global_blend = sum(wi * o for wi, o in zip(best_weights, oof_stack))
global_score = balanced_accuracy_score(y, global_blend.argmax(axis=1))

print("global blend: " + "  ".join(f"{m}={w:.2f}" for m, w in zip(model_names, best_weights)))
print(f"global OOF balanced_acc: {global_score:.5f}")
for cls_idx, cls_name in enumerate(LABELS):
    print("  " + cls_name + ": " + "  ".join(f"{m}={W[i, cls_idx]:.2f}" for i, m in enumerate(model_names)))
print(f"selected: {'per-class' if use_per_class else 'global'}    OOF balanced_acc: {best_blend_score:.5f}")

def apply_temperature(probs, T):
    eps = 1e-15
    logp = np.log(np.clip(probs, eps, 1.0))
    out = np.exp(logp / T)
    out /= out.sum(axis=1, keepdims=True)
    return out

blend_score_raw = balanced_accuracy_score(y, oof_blend.argmax(axis=1))
best_T = 1.0
temp_score = blend_score_raw
for T in np.linspace(0.6, 1.6, 21):
    s = balanced_accuracy_score(y, apply_temperature(oof_blend, T).argmax(axis=1))
    if s > temp_score:
        temp_score, best_T = s, float(T)
for T in np.linspace(max(0.5, best_T - 0.08), min(2.0, best_T + 0.08), 17):
    s = balanced_accuracy_score(y, apply_temperature(oof_blend, T).argmax(axis=1))
    if s > temp_score:
        temp_score, best_T = s, float(T)

use_temp = temp_score > blend_score_raw
if use_temp:
    oof_blend = apply_temperature(oof_blend, best_T)
    test_pred_blend = apply_temperature(test_pred_blend, best_T)
    best_blend_score = temp_score
print(f"temperature T={best_T:.3f}    OOF: {temp_score:.5f}")
print(f"calibration: {'temperature' if use_temp else 'none'}    blend OOF: {best_blend_score:.5f}")

QSO_IDX = label_to_idx["QSO"]
STAR_IDX = label_to_idx["STAR"]
GALAXY_IDX = label_to_idx["GALAXY"]

def apply_gated_scale(probs, s_qso, s_star, qso_thresh):
    out = probs.copy()
    mask = probs[:, QSO_IDX] < qso_thresh
    out[mask, QSO_IDX] = probs[mask, QSO_IDX] * s_qso
    out[mask, STAR_IDX] = probs[mask, STAR_IDX] * s_star
    out[mask] /= out[mask].sum(axis=1, keepdims=True)
    return out

def apply_tri_z_binned_scale(probs, z_vals, z_lo, z_hi, sq0, st0, sq1, st1, sq2, st2, qso_thresh):
    out = probs.copy()
    low = z_vals < z_lo
    mid = (z_vals >= z_lo) & (z_vals < z_hi)
    high = z_vals >= z_hi
    for mask, sq, st in [(low, sq0, st0), (mid, sq1, st1), (high, sq2, st2)]:
        if mask.any():
            out[mask] = apply_gated_scale(probs[mask], sq, st, qso_thresh)
    return out

def eval_tri_z_binned(probs, target, z_vals, z_lo, z_hi, sq0, st0, sq1, st1, sq2, st2, qso_thresh):
    pred = apply_tri_z_binned_scale(probs, z_vals, z_lo, z_hi, sq0, st0, sq1, st1, sq2, st2, qso_thresh).argmax(axis=1)
    return balanced_accuracy_score(target, pred)

def search_tri_z_binned_bin(probs, target, z_vals, z_lo, z_hi, bin_idx, g1, g2, sq0, st0, sq1, st1, sq2, st2, qso_thresh, best_score):
    for s1 in g1:
        for s2 in g2:
            if bin_idx == 0:
                sc = eval_tri_z_binned(probs, target, z_vals, z_lo, z_hi, s1, s2, sq1, st1, sq2, st2, qso_thresh)
                if sc > best_score:
                    best_score, sq0, st0 = sc, s1, s2
            elif bin_idx == 1:
                sc = eval_tri_z_binned(probs, target, z_vals, z_lo, z_hi, sq0, st0, s1, s2, sq2, st2, qso_thresh)
                if sc > best_score:
                    best_score, sq1, st1 = sc, s1, s2
            else:
                sc = eval_tri_z_binned(probs, target, z_vals, z_lo, z_hi, sq0, st0, sq1, st1, s1, s2, qso_thresh)
                if sc > best_score:
                    best_score, sq2, st2 = sc, s1, s2
    return sq0, st0, sq1, st1, sq2, st2, best_score

def apply_binned_gated_scale(probs, bin_vals, bin_thresh, s_qso_lo, s_star_lo, s_qso_hi, s_star_hi, qso_thresh):
    out = probs.copy()
    low = bin_vals < bin_thresh
    high = ~low
    if low.any():
        out[low] = apply_gated_scale(probs[low], s_qso_lo, s_star_lo, qso_thresh)
    if high.any():
        out[high] = apply_gated_scale(probs[high], s_qso_hi, s_star_hi, qso_thresh)
    return out

def eval_binned(probs, target, bin_vals, bin_thresh, sq_lo, st_lo, sq_hi, st_hi, qso_thresh):
    pred = apply_binned_gated_scale(probs, bin_vals, bin_thresh, sq_lo, st_lo, sq_hi, st_hi, qso_thresh).argmax(axis=1)
    return balanced_accuracy_score(target, pred)

def search_binned_bin(probs, target, bin_vals, bin_thresh, low_bin, g1, g2, sq_lo, st_lo, sq_hi, st_hi, qso_thresh, best_score):
    best_sq_l, best_st_l, best_sq_h, best_st_h = sq_lo, st_lo, sq_hi, st_hi
    for s1 in g1:
        for s2 in g2:
            if low_bin:
                sc = eval_binned(probs, target, bin_vals, bin_thresh, s1, s2, best_sq_h, best_st_h, qso_thresh)
                if sc > best_score:
                    best_score, best_sq_l, best_st_l = sc, s1, s2
            else:
                sc = eval_binned(probs, target, bin_vals, bin_thresh, best_sq_l, best_st_l, s1, s2, qso_thresh)
                if sc > best_score:
                    best_score, best_sq_h, best_st_h = sc, s1, s2
    return best_sq_l, best_st_l, best_sq_h, best_st_h, best_score

def apply_tri_gated_scale(probs, s_galaxy, s_qso, s_star, qso_thresh):
    out = probs.copy()
    mask = probs[:, QSO_IDX] < qso_thresh
    out[mask, GALAXY_IDX] = probs[mask, GALAXY_IDX] * s_galaxy
    out[mask, QSO_IDX] = probs[mask, QSO_IDX] * s_qso
    out[mask, STAR_IDX] = probs[mask, STAR_IDX] * s_star
    out[mask] /= out[mask].sum(axis=1, keepdims=True)
    return out

def search_galaxy_scalar(probs, target, s_qso, s_star, qso_thresh, grid, best_sg, best_score):
    for sg in grid:
        pred = apply_tri_gated_scale(probs, sg, s_qso, s_star, qso_thresh).argmax(axis=1)
        sc = balanced_accuracy_score(target, pred)
        if sc > best_score:
            best_score, best_sg = sc, sg
    return best_sg, best_score

def apply_gs_fix(probs, s_galaxy, s_star, qso_thresh, margin):
    out = probs.copy()
    qso_ok = probs[:, QSO_IDX] < qso_thresh
    competitive = (probs[:, STAR_IDX] + margin >= probs[:, GALAXY_IDX]) & (
        probs[:, GALAXY_IDX] + margin >= probs[:, STAR_IDX]
    )
    mask = qso_ok & competitive
    out[mask, GALAXY_IDX] = probs[mask, GALAXY_IDX] * s_galaxy
    out[mask, STAR_IDX] = probs[mask, STAR_IDX] * s_star
    out[mask] /= out[mask].sum(axis=1, keepdims=True)
    return out

def search_gs_fix(probs, target, g_g, g_s, margins, qso_thresh, best):
    best_sg, best_ss, best_margin, best_score = best
    for margin in margins:
        for sg in g_g:
            for ss in g_s:
                pred = apply_gs_fix(probs, sg, ss, qso_thresh, margin).argmax(axis=1)
                sc = balanced_accuracy_score(target, pred)
                if sc > best_score:
                    best_score, best_sg, best_ss, best_margin = sc, sg, ss, margin
    return best_sg, best_ss, best_margin, best_score

def search_ungated(probs, target, g1, g2, best):
    best_s1, best_s2, best_score = best
    for s1 in g1:
        for s2 in g2:
            pred = apply_gated_scale(probs, s1, s2, 1.0).argmax(axis=1)
            s = balanced_accuracy_score(target, pred)
            if s > best_score:
                best_score, best_s1, best_s2 = s, s1, s2
    return best_s1, best_s2, best_score

def search_gated(probs, target, g1, g2, t_grid, best):
    best_s1, best_s2, best_t, best_score = best
    for t in t_grid:
        for s1 in g1:
            for s2 in g2:
                pred = apply_gated_scale(probs, s1, s2, t).argmax(axis=1)
                s = balanced_accuracy_score(target, pred)
                if s > best_score:
                    best_score, best_s1, best_s2, best_t = s, s1, s2, t
    return best_s1, best_s2, best_t, best_score

base_scaled_score = balanced_accuracy_score(y, oof_blend.argmax(axis=1))
coarse = np.linspace(0.2, 3.0, 29)
thresh_coarse = np.linspace(0.15, 0.85, 15)

u1, u2, ungated_score = search_ungated(oof_blend, y, coarse, coarse, (1.0, 1.0, base_scaled_score))
uf1 = np.linspace(max(0.05, u1 - 0.2), u1 + 0.2, 41)
uf2 = np.linspace(max(0.05, u2 - 0.2), u2 + 0.2, 41)
u1, u2, ungated_score = search_ungated(oof_blend, y, uf1, uf2, (u1, u2, ungated_score))

g1, g2, gt, gated_score = search_gated(oof_blend, y, coarse, coarse, thresh_coarse, (1.0, 1.0, 0.5, base_scaled_score))
gf1 = np.linspace(max(0.05, g1 - 0.2), g1 + 0.2, 41)
gf2 = np.linspace(max(0.05, g2 - 0.2), g2 + 0.2, 41)
gt_fine = np.linspace(max(0.05, gt - 0.1), min(0.95, gt + 0.1), 21)
g1, g2, gt, gated_score = search_gated(oof_blend, y, gf1, gf2, gt_fine, (g1, g2, gt, gated_score))

tri_sg, tri_score = 1.0, gated_score
tri_sg, tri_score = search_galaxy_scalar(oof_blend, y, g1, g2, gt, np.linspace(0.75, 1.6, 18), 1.0, gated_score)
tri_sg, tri_score = search_galaxy_scalar(
    oof_blend, y, g1, g2, gt, np.linspace(max(0.5, tri_sg - 0.12), tri_sg + 0.12, 25), tri_sg, tri_score,
)

use_gated = gated_score >= ungated_score
use_tri = tri_score >= max(gated_score, ungated_score)
if use_tri:
    best_s1, best_s2, best_sg, best_qso_thresh, best_scaled_score = g1, g2, tri_sg, gt, tri_score
elif use_gated:
    best_s1, best_s2, best_sg, best_qso_thresh, best_scaled_score = g1, g2, 1.0, gt, gated_score
else:
    best_s1, best_s2, best_sg, best_qso_thresh, best_scaled_score = u1, u2, 1.0, 1.0, ungated_score

def apply_global_scale(probs):
    if use_tri:
        return apply_tri_gated_scale(probs, best_sg, best_s1, best_s2, best_qso_thresh)
    return apply_gated_scale(probs, best_s1, best_s2, best_qso_thresh)

z_train = X["redshift"].to_numpy(dtype=np.float64)
z_test = X_test["redshift"].to_numpy(dtype=np.float64)
z_grid = np.unique(np.percentile(z_train, [10, 25, 33, 50, 67, 75, 90]))

best_z_thresh = float(np.median(z_train))
b_lo_qso, b_lo_star = best_s1, best_s2
b_hi_qso, b_hi_star = best_s1, best_s2
binned_score_val = best_scaled_score

for z_thresh in z_grid:
    sq_l, st_l, sq_h, st_h, sc = search_binned_bin(
        oof_blend, y, z_train, z_thresh, True, coarse, coarse,
        best_s1, best_s2, best_s1, best_s2, best_qso_thresh, binned_score_val,
    )
    sq_l, st_l, sq_h, st_h, sc = search_binned_bin(
        oof_blend, y, z_train, z_thresh, False, coarse, coarse,
        sq_l, st_l, best_s1, best_s2, best_qso_thresh, sc,
    )
    if sc > binned_score_val:
        binned_score_val, best_z_thresh = sc, float(z_thresh)
        b_lo_qso, b_lo_star, b_hi_qso, b_hi_star = sq_l, st_l, sq_h, st_h

bf1_lo = np.linspace(max(0.05, b_lo_qso - 0.2), b_lo_qso + 0.2, 41)
bf2_lo = np.linspace(max(0.05, b_lo_star - 0.2), b_lo_star + 0.2, 41)
bf1_hi = np.linspace(max(0.05, b_hi_qso - 0.2), b_hi_qso + 0.2, 41)
bf2_hi = np.linspace(max(0.05, b_hi_star - 0.2), b_hi_star + 0.2, 41)

sq_l, st_l, sq_h, st_h, sc = search_binned_bin(
    oof_blend, y, z_train, best_z_thresh, True, bf1_lo, bf2_lo,
    b_lo_qso, b_lo_star, b_hi_qso, b_hi_star, best_qso_thresh, binned_score_val,
)
sq_l, st_l, sq_h, st_h, sc = search_binned_bin(
    oof_blend, y, z_train, best_z_thresh, False, bf1_hi, bf2_hi,
    sq_l, st_l, b_hi_qso, b_hi_star, best_qso_thresh, sc,
)
if sc > binned_score_val:
    binned_score_val = sc
    b_lo_qso, b_lo_star, b_hi_qso, b_hi_star = sq_l, st_l, sq_h, st_h

z_lo_grid = np.unique(np.percentile(z_train, [25, 33, 50]))
z_hi_grid = np.unique(np.percentile(z_train, [67, 75, 90]))
best_z_lo = float(z_lo_grid[0])
best_z_hi = float(z_hi_grid[-1])
tz0_q, tz0_s = b_lo_qso, b_lo_star
tz1_q, tz1_s = best_s1, best_s2
tz2_q, tz2_s = b_hi_qso, b_hi_star
tri_z_score = binned_score_val

for z_lo in z_lo_grid:
    for z_hi in z_hi_grid:
        if z_hi <= z_lo:
            continue
        q0, s0 = tz0_q, tz0_s
        q1, s1 = tz1_q, tz1_s
        q2, s2 = tz2_q, tz2_s
        sc = tri_z_score
        for bi in (0, 1, 2):
            q0, s0, q1, s1, q2, s2, sc = search_tri_z_binned_bin(
                oof_blend, y, z_train, z_lo, z_hi, bi, coarse, coarse,
                q0, s0, q1, s1, q2, s2, best_qso_thresh, sc,
            )
        if sc > tri_z_score:
            tri_z_score, best_z_lo, best_z_hi = sc, float(z_lo), float(z_hi)
            tz0_q, tz0_s, tz1_q, tz1_s, tz2_q, tz2_s = q0, s0, q1, s1, q2, s2

tf0_q = np.linspace(max(0.05, tz0_q - 0.2), tz0_q + 0.2, 41)
tf0_s = np.linspace(max(0.05, tz0_s - 0.2), tz0_s + 0.2, 41)
tf1_q = np.linspace(max(0.05, tz1_q - 0.2), tz1_q + 0.2, 41)
tf1_s = np.linspace(max(0.05, tz1_s - 0.2), tz1_s + 0.2, 41)
tf2_q = np.linspace(max(0.05, tz2_q - 0.2), tz2_q + 0.2, 41)
tf2_s = np.linspace(max(0.05, tz2_s - 0.2), tz2_s + 0.2, 41)

for bi, fq, fs in [(0, tf0_q, tf0_s), (1, tf1_q, tf1_s), (2, tf2_q, tf2_s)]:
    tz0_q, tz0_s, tz1_q, tz1_s, tz2_q, tz2_s, tri_z_score = search_tri_z_binned_bin(
        oof_blend, y, z_train, best_z_lo, best_z_hi, bi, fq, fs,
        tz0_q, tz0_s, tz1_q, tz1_s, tz2_q, tz2_s, best_qso_thresh, tri_z_score,
    )

scale_candidates = [
    (
        "global",
        best_scaled_score,
        apply_global_scale(oof_blend),
        apply_global_scale(test_pred_blend),
    ),
    (
        "z2_binned",
        binned_score_val,
        apply_binned_gated_scale(
            oof_blend, z_train, best_z_thresh, b_lo_qso, b_lo_star, b_hi_qso, b_hi_star, best_qso_thresh,
        ),
        apply_binned_gated_scale(
            test_pred_blend, z_test, best_z_thresh, b_lo_qso, b_lo_star, b_hi_qso, b_hi_star, best_qso_thresh,
        ),
    ),
    (
        "z3_binned",
        tri_z_score,
        apply_tri_z_binned_scale(
            oof_blend, z_train, best_z_lo, best_z_hi,
            tz0_q, tz0_s, tz1_q, tz1_s, tz2_q, tz2_s, best_qso_thresh,
        ),
        apply_tri_z_binned_scale(
            test_pred_blend, z_test, best_z_lo, best_z_hi,
            tz0_q, tz0_s, tz1_q, tz1_s, tz2_q, tz2_s, best_qso_thresh,
        ),
    ),
]
scale_label, scaled_score, oof_scaled, test_scaled = max(scale_candidates, key=lambda x: x[1])
margin_coarse = [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.22, 0.30]
galaxy_coarse = np.linspace(0.85, 2.2, 14)
star_coarse = np.linspace(0.45, 1.15, 15)

gs_sg, gs_ss, gs_margin, gs_score = search_gs_fix(
    oof_scaled, y, galaxy_coarse, star_coarse, margin_coarse, best_qso_thresh, (1.0, 1.0, 0.0, scaled_score),
)
gf_g = np.linspace(max(0.5, gs_sg - 0.25), gs_sg + 0.25, 41)
gf_s = np.linspace(max(0.3, gs_ss - 0.15), min(1.5, gs_ss + 0.15), 41)
margin_fine = np.linspace(max(0.02, gs_margin - 0.04), gs_margin + 0.04, 9)
gs_sg, gs_ss, gs_margin, gs_score = search_gs_fix(
    oof_scaled, y, gf_g, gf_s, margin_fine, best_qso_thresh, (gs_sg, gs_ss, gs_margin, gs_score),
)

use_gs = gs_score > scaled_score
if use_gs:
    best_scaled_score = gs_score
    oof_final = apply_gs_fix(oof_scaled, gs_sg, gs_ss, best_qso_thresh, gs_margin)
    test_pred_final = apply_gs_fix(test_scaled, gs_sg, gs_ss, best_qso_thresh, gs_margin)
    scale_label = scale_label + "+gs_fix"
else:
    best_scaled_score = scaled_score
    oof_final = oof_scaled
    test_pred_final = test_scaled

print(f"ungated scalars: QSO={u1:.3f}  STAR={u2:.3f}    OOF: {ungated_score:.5f}")
print(f"gated scalars: QSO={g1:.3f}  STAR={g2:.3f}  qso_thresh={gt:.3f}    OOF: {gated_score:.5f}")
print(f"tri scalars: GALAXY={tri_sg:.3f}  QSO={g1:.3f}  STAR={g2:.3f}  qso_thresh={gt:.3f}    OOF: {tri_score:.5f}")
print(
    f"global selected: {'tri' if use_tri else ('gated' if use_gated else 'ungated')}    "
    f"GALAXY={best_sg:.3f}  QSO={best_s1:.3f}  STAR={best_s2:.3f}  qso_thresh={best_qso_thresh:.3f}"
)
print(
    f"z2_binned z<{best_z_thresh:.4f}: QSO={b_lo_qso:.3f} STAR={b_lo_star:.3f}  |  "
    f"z>={best_z_thresh:.4f}: QSO={b_hi_qso:.3f} STAR={b_hi_star:.3f}    OOF: {binned_score_val:.5f}"
)
print(
    f"z3 lo/mid/hi (z<{best_z_lo:.4f}, z>={best_z_hi:.4f}): "
    f"lo QSO={tz0_q:.3f} STAR={tz0_s:.3f}  mid QSO={tz1_q:.3f} STAR={tz1_s:.3f}  "
    f"hi QSO={tz2_q:.3f} STAR={tz2_s:.3f}    OOF: {tri_z_score:.5f}"
)
print(
    f"gs_fix: GALAXY={gs_sg:.3f}  STAR={gs_ss:.3f}  margin={gs_margin:.3f}    OOF: {gs_score:.5f}"
)
print(f"selected: {scale_label}")
print(f"OOF balanced_acc after scaling: {best_scaled_score:.5f}    (delta: {best_scaled_score - base_scaled_score:+.5f})")

global blend: lgb=0.31  lgb_mid=0.30  lgb_deep=0.09  cb=0.24  xgb=0.06
global OOF balanced_acc: 0.96592
  GALAXY: lgb=0.36  lgb_mid=0.31  lgb_deep=0.04  cb=0.22  xgb=0.07
  QSO: lgb=0.09  lgb_mid=0.00  lgb_deep=0.20  cb=0.65  xgb=0.06
  STAR: lgb=0.30  lgb_mid=0.39  lgb_deep=0.01  cb=0.25  xgb=0.05
selected: per-class    OOF balanced_acc: 0.96605
temperature T=1.000    OOF: 0.96605
calibration: none    blend OOF: 0.96605
ungated scalars: QSO=0.950  STAR=1.000    OOF: 0.96606
gated scalars: QSO=0.610  STAR=1.000  qso_thresh=0.490    OOF: 0.96608
tri scalars: GALAXY=1.000  QSO=0.610  STAR=1.000  qso_thresh=0.490    OOF: 0.96608
global selected: tri    GALAXY=1.000  QSO=0.610  STAR=1.000  qso_thresh=0.490
z2_binned z<0.1811: QSO=0.610 STAR=1.000  |  z>=0.1811: QSO=0.820 STAR=1.480    OOF: 0.96611
z3 lo/mid/hi (z<0.1811, z>=1.8199): lo QSO=0.610 STAR=1.000  mid QSO=0.610 STAR=1.000  hi QSO=0.820 STAR=1.480    OOF: 0.96611
gs_fix: GALAXY=1.150  STAR=1.150  margin=0.022    OOF: 0.96611
selec

In [112]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_final = oof_final.argmax(axis=1)

cm = confusion_matrix(y, y_pred_final)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
print("confusion matrix (counts):")
print(cm_df.to_string())
print()

cm_norm = cm / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
print("row-normalized (diagonal = per-class recall):")
print(cm_norm_df.round(5).to_string())
print()

per_class_recall = np.diag(cm_norm)
print("per-class recall (mean = balanced accuracy):")
for label, r in zip(LABELS, per_class_recall):
    print(f"  {label}: {r:.5f}")
print(f"  mean:        {per_class_recall.mean():.5f}")
print()

off_diag = []
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if i != j:
            off_diag.append((LABELS[i], LABELS[j], int(cm[i, j]), cm_norm[i, j]))
off_diag.sort(key=lambda t: -t[2])
print("top off-diagonal errors (true -> pred):")
for true_l, pred_l, count, rate in off_diag:
    print(f"  {true_l:>6} -> {pred_l:<6}  {count:>8}    ({rate:.4%} of true {true_l})")
print()

print("classification report:")
print(classification_report(y, y_pred_final, target_names=LABELS, digits=5))

if "fi_lgb" in globals():
    fi_df = pd.DataFrame({"feature": ALL_FEATURES, "gain": fi_lgb}).sort_values("gain", ascending=False).reset_index(drop=True)
    total_gain = fi_df["gain"].sum()
    fi_df["pct_of_total"] = (fi_df["gain"] / total_gain * 100).round(3)
    fi_df["cumulative_pct"] = fi_df["pct_of_total"].cumsum().round(3)
    print("LightGBM feature importance (gain, averaged across all folds):")
    print(fi_df.to_string(index=False))

    weak = fi_df[fi_df["pct_of_total"] < 0.5]["feature"].tolist()
    if weak:
        print(f"\nfeatures contributing <0.5% of total gain (drop candidates): {weak}")
else:
    print("feature importance not available -- rerun the LightGBM CV cell to populate fi_lgb")

confusion matrix (counts):
             pred_GALAXY  pred_QSO  pred_STAR
true_GALAXY       359691      5796      11993
true_QSO            1645    114215       1283
true_STAR           2124       321      80279

row-normalized (diagonal = per-class recall):
             pred_GALAXY  pred_QSO  pred_STAR
true_GALAXY      0.95287   0.01535    0.03177
true_QSO         0.01404   0.97500    0.01095
true_STAR        0.02568   0.00388    0.97044

per-class recall (mean = balanced accuracy):
  GALAXY: 0.95287
  QSO: 0.97500
  STAR: 0.97044
  mean:        0.96611

top off-diagonal errors (true -> pred):
  GALAXY -> STAR       11993    (3.1771% of true GALAXY)
  GALAXY -> QSO         5796    (1.5354% of true GALAXY)
    STAR -> GALAXY      2124    (2.5676% of true STAR)
     QSO -> GALAXY      1645    (1.4043% of true QSO)
     QSO -> STAR        1283    (1.0952% of true QSO)
    STAR -> QSO          321    (0.3880% of true STAR)

classification report:
              precision    recall  f1-score

In [113]:
pred_labels = np.array([idx_to_label[i] for i in test_pred_final.argmax(axis=1)])
submission = pd.DataFrame({"id": test["id"], "class": pred_labels})
submission.to_csv("submission.csv", index=False)

print(f"submission shape: {submission.shape}")
print()
print("predicted class distribution:")
print(submission["class"].value_counts(normalize=True).round(4).to_string())
print()
submission.head()

submission shape: (247435, 2)

predicted class distribution:
class
GALAXY    0.6303
QSO       0.2081
STAR      0.1615



,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
